In [1]:
from common_utils import vector_from_points, angle_between_vectors

In [2]:
from common_utils import save_dataframe_to_csv

In [3]:
import pandas as pd
import numpy as np

In [4]:
def compute_shoulder_angle(row, side="left"):
    """
    Angle at shoulder formed by: hip - shoulder - elbow
    """
    hip = row[[f"{side}_hip_x", f"{side}_hip_y", f"{side}_hip_z"]].values
    shoulder = row[[f"{side}_shoulder_x", f"{side}_shoulder_y", f"{side}_shoulder_z"]].values
    elbow = row[[f"{side}_elbow_x", f"{side}_elbow_y", f"{side}_elbow_z"]].values
    vec1 = vector_from_points(hip, shoulder)
    vec2 = vector_from_points(elbow, shoulder)
    return angle_between_vectors(vec1, vec2)

def compute_elbow_angle(row, side="left"):
    """
    Angle at elbow formed by: shoulder - elbow - wrist
    """
    shoulder = row[[f"{side}_shoulder_x", f"{side}_shoulder_y", f"{side}_shoulder_z"]].values
    elbow = row[[f"{side}_elbow_x", f"{side}_elbow_y", f"{side}_elbow_z"]].values
    wrist = row[[f"{side}_wrist_x", f"{side}_wrist_y", f"{side}_wrist_z"]].values
    vec1 = vector_from_points(shoulder, elbow)
    vec2 = vector_from_points(wrist, elbow)
    return angle_between_vectors(vec1, vec2)

def compute_hip_angle(row, side="left"):
    """
    Angle at hip formed by: shoulder - hip - knee
    """
    shoulder = row[[f"{side}_shoulder_x", f"{side}_shoulder_y", f"{side}_shoulder_z"]].values
    hip = row[[f"{side}_hip_x", f"{side}_hip_y", f"{side}_hip_z"]].values
    knee = row[[f"{side}_knee_x", f"{side}_knee_y", f"{side}_knee_z"]].values
    vec1 = vector_from_points(shoulder, hip)
    vec2 = vector_from_points(knee, hip)
    return angle_between_vectors(vec1, vec2)

def compute_knee_angle(row, side="left"):
    """
    Angle at knee formed by: hip - knee - ankle
    """
    hip = row[[f"{side}_hip_x", f"{side}_hip_y", f"{side}_hip_z"]].values
    knee = row[[f"{side}_knee_x", f"{side}_knee_y", f"{side}_knee_z"]].values
    ankle = row[[f"{side}_ankle_x", f"{side}_ankle_y", f"{side}_ankle_z"]].values
    vec1 = vector_from_points(hip, knee)
    vec2 = vector_from_points(ankle, knee)
    return angle_between_vectors(vec1, vec2)

In [5]:
# =========================
# Add All Plank Angles to Dataset
# =========================


def add_plank_angles(df):
    """
    Adds plank-specific angles to the dataframe.
    Angles added per side: shoulder_angle, elbow_angle, hip_angle, knee_angle
    df: pandas DataFrame with selective landmarks
    Returns df with new angle columns and a list of angle column names
    """
    angle_columns = []
    for side in ["left", "right"]:
        # Shoulder angle: hip - shoulder - elbow
        df[f"{side}_shoulder_angle"] = df.apply(lambda row: compute_shoulder_angle(row, side), axis=1)
        # Elbow angle: shoulder - elbow - wrist
        df[f"{side}_elbow_angle"] = df.apply(lambda row: compute_elbow_angle(row, side), axis=1)
        # Hip angle: shoulder - hip - knee
        df[f"{side}_hip_angle"] = df.apply(lambda row: compute_hip_angle(row, side), axis=1)
        # Knee angle: hip - knee - ankle
        df[f"{side}_knee_angle"] = df.apply(lambda row: compute_knee_angle(row, side), axis=1)

        angle_columns += [f"{side}_shoulder_angle", f"{side}_elbow_angle", f"{side}_hip_angle", f"{side}_knee_angle"]

    return df, angle_columns

In [6]:
df = pd.read_csv("plank_cleaned_train.csv")  # Dataset with selective landmarks (all lowercase)
df, angle_cols = add_plank_angles(df)
print(df.head())
print("New angle columns:", angle_cols)

  label    nose_x    nose_y    nose_z    nose_v  left_shoulder_x  \
0     C  0.792141  0.585212 -0.067640  0.999518         0.682632   
1     C  0.792153  0.585208 -0.067657  0.999518         0.682624   
2     C  0.792160  0.585211 -0.067602  0.999519         0.682617   
3     C  0.792171  0.585225 -0.067480  0.999520         0.682611   
4     C  0.792179  0.585240 -0.067460  0.999522         0.682607   

   left_shoulder_y  left_shoulder_z  left_shoulder_v  right_shoulder_x  ...  \
0         0.539670         0.270455         0.996880          0.688922  ...   
1         0.539675         0.270204         0.996868          0.688926  ...   
2         0.539684         0.270091         0.996864          0.688927  ...   
3         0.539690         0.270020         0.996866          0.688929  ...   
4         0.539697         0.269914         0.996867          0.688932  ...   

   right_foot_index_z  right_foot_index_v  left_shoulder_angle  \
0           -0.230562            0.952622         

In [7]:
# Save dataset using common function
save_dataframe_to_csv(df, "plank_with_angles.csv")

DataFrame successfully saved to: plank_with_angles.csv
